In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from sklearnex import patch_sklearn
patch_sklearn()
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix
from reader import prepare_qsm_dataset
from util import seed_everything, mask_crop as mask_crop_fn
from train import calibrate_balanced

# --- CONFIG ---
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
N_PCA_COMPONENTS = 16 
N_FOLDS = 5 
NEG_WEIGHT = 12.0  
GAMMA = 12.0
JITTER_STD = 0.1 
TARGET_DIM = 128
IMG_AUG_STD = 0.05
PSEUDO_THRESHOLD = 0.0  # Use all unlabeled data
seed_everything(0)

# --- STANDALONE METRICS HELPER ---
def get_m(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    sens = recall_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    auc = roc_auc_score(y_true, y_prob)
    return [acc, sens, spec, auc]

class PassThrough(nn.Module):
    def forward(self, x, **kwargs): return x

# ============================================================
# MODELS
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets, weight=None):
        bce_loss = F.binary_cross_entropy(inputs, targets, weight=weight, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

class ClinicalTransformer(nn.Module):
    def __init__(self, n_inputs, embed_dim=32, n_heads=4, n_layers=2):
        super().__init__()
        self.tau = nn.Parameter(torch.tensor(1024.0)) 
        self.n_inputs = n_inputs
        self.embedding = nn.Linear(1, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=n_heads, 
            dim_feedforward=embed_dim*2,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc = nn.Linear(embed_dim * n_inputs, 1)
    def forward(self, x, return_logit=False, **kwargs):
        if x.ndim == 1: 
            x = x.unsqueeze(0)
        if x.shape[1] == self.n_inputs:
            indices = torch.arange(self.n_inputs).to(x.device).float()
            omega = torch.exp(-indices / self.tau)
            x = x*omega
        x = x.unsqueeze(-1)          
        x = self.embedding(x)        
        x = x.permute(1, 0, 2)       
        x = self.transformer(x)      
        x = x.permute(1, 0, 2).flatten(1)
        logit = self.fc(x).squeeze()
        if logit.ndim == 0: 
            logit = logit.unsqueeze(0) 
        return logit if return_logit else torch.sigmoid(logit)

class ResidualWrapper(nn.Module):
    def __init__(self, m_ct, m_res):
        super().__init__()
        self.m_ct = m_ct
        self.m_res = m_res
    def forward(self, x_joint, **kwargs):
        x_pca = x_joint[:, :N_PCA_COMPONENTS]
        x_clin = x_joint[:, N_PCA_COMPONENTS:]
        l_clin = self.m_ct(x_clin, return_logit=True)
        l_res = self.m_res(x_pca, return_logit=True)
        return torch.sigmoid(l_clin + l_res)

class SpatialViT(nn.Module):
    def __init__(self, img_size=128, patch_size=16, embed_dim=32, n_heads=4, n_layers=2):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Linear(patch_size * patch_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*2, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc = nn.Linear(embed_dim * self.n_patches, 1)

    def forward(self, x, return_logit=False):
        b, c, h, w = x.shape
        x = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        x = x.contiguous().view(b, self.n_patches, -1)
        x = self.proj(x) + self.pos_embed
        x = x.permute(1, 0, 2) 
        x = self.transformer(x).permute(1, 0, 2).flatten(1)
        logit = self.fc(x).squeeze()
        if logit.ndim == 0: logit = logit.unsqueeze(0)
        return logit if return_logit else torch.sigmoid(logit)

def robust_flatten(img_np, target_dim=TARGET_DIM):
    t = torch.from_numpy(img_np).float().unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(t, size=(target_dim, target_dim), mode='bilinear', align_corners=False)
    return resized.numpy().flatten()

# ============================================================
# DATA PREPARATION
# ============================================================
dataset_msw = prepare_qsm_dataset('MSW', '/media/mts_dbs/dbs/all/nii/qsm_115/im', '/media/mts_dbs/dbs/all/nii/seg_ps/', '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv', 'msw_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)
dataset_chh = prepare_qsm_dataset('CHH', '/media/mts_dbs/chh/nii/qsm/', '/media/mts_dbs/chh/roi/', '/media/mts_dbs/chh/xlsx/chh_subjects_table1_20240729.csv', 'chh_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)

def get_slice_level_data(dataset, include_unlabeled=False):
    all_imgs, all_clins, all_lbls, subj_map = [], [], [], []
    for i in range(len(dataset)):
        img, clin, lbl, _ = dataset[i]
        if (i % 72) in [0]: continue
        if (lbl != -1) or (include_unlabeled and lbl == -1):
            img_np = img.numpy().squeeze()
            all_imgs.append(robust_flatten(img_np))
            all_clins.append(clin.numpy()); all_lbls.append(lbl); subj_map.append(i)
    return np.array(all_imgs), np.array(all_clins), np.array(all_lbls), np.array(subj_map)

X_full_slices, X_full_clin, y_full_slices, full_subj_map = get_slice_level_data(dataset_msw, include_unlabeled=True)
labeled_mask = (y_full_slices != -1)
X_tr_slices, X_tr_clin, y_tr_slices, tr_subj_map = X_full_slices[labeled_mask], X_full_clin[labeled_mask], y_full_slices[labeled_mask], full_subj_map[labeled_mask]
X_un_slices, X_un_clin, y_un_slices, un_subj_map = X_full_slices[~labeled_mask], X_full_clin[~labeled_mask], y_full_slices[~labeled_mask], full_subj_map[~labeled_mask]
X_te_slices, X_te_clin, y_te_slices, te_subj_map = get_slice_level_data(dataset_chh)

img_scaler = StandardScaler().fit(X_tr_slices)
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0, whiten=True).fit(img_scaler.transform(X_tr_slices))
X_tr_pca = pca.transform(img_scaler.transform(X_tr_slices))
X_te_pca = pca.transform(img_scaler.transform(X_te_slices))
X_un_pca = pca.transform(img_scaler.transform(X_un_slices)) if len(X_un_slices) > 0 else []

# ============================================================
# CROSS-VALIDATION LOOP
# ============================================================
unique_subjs = np.unique(tr_subj_map)
y_unique = np.array([y_tr_slices[tr_subj_map == s][0] for s in unique_subjs])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)

fold_preds_ct, fold_preds_sp, fold_preds_spatial = [], [], []
fold_ths_ct, fold_ths_sp, fold_ths_spatial = [], [], []
criterion = FocalLoss(gamma=GAMMA)

for fold, (t_subj_idx, v_subj_idx) in enumerate(skf.split(unique_subjs, y_unique)):
    t_idx_slices = np.isin(tr_subj_map, unique_subjs[t_subj_idx])
    v_idx_slices = np.isin(tr_subj_map, unique_subjs[v_subj_idx])
    X_tr_clin_subjs = np.array([X_tr_clin[tr_subj_map == s][0] for s in unique_subjs[t_subj_idx]])
    y_tr_subjs = y_unique[t_subj_idx]
    
    scaler_c = StandardScaler().fit(X_tr_clin_subjs)
    xt_clin_subjs = torch.tensor(scaler_c.transform(X_tr_clin_subjs), dtype=torch.float32).to(device)
    yt_subjs = torch.tensor(y_tr_subjs, dtype=torch.float32).to(device)

    # --- 1. Clinical Transformer ---
    m_ct = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    opt_ct = optim.Adam(m_ct.parameters(), lr=1e-4)
    for epoch in range(100):
        m_ct.train(); opt_ct.zero_grad()
        c_jitter = torch.randn_like(xt_clin_subjs) * JITTER_STD
        w = torch.where(yt_subjs == 0, torch.tensor(NEG_WEIGHT).to(device), torch.tensor(1.0).to(device))
        loss = criterion(m_ct(xt_clin_subjs + c_jitter), yt_subjs, weight=w)
        loss.backward(); opt_ct.step()

    # --- 2. Spectral Residual ---
    for param in m_ct.parameters(): param.requires_grad = False
    m_ct.eval()
    m_res = ClinicalTransformer(n_inputs=N_PCA_COMPONENTS).to(device)
    opt_res = optim.Adam(m_res.parameters(), lr=1e-4)

    xt_pca_slices = torch.tensor(X_tr_pca[t_idx_slices], dtype=torch.float32).to(device)
    xt_clin_slices = torch.tensor(scaler_c.transform(X_tr_clin[t_idx_slices]), dtype=torch.float32).to(device)
    yt_slices = torch.tensor(y_tr_slices[t_idx_slices], dtype=torch.float32).to(device)

    for epoch in range(150):
        m_res.train(); opt_res.zero_grad()
        i_aug = torch.randn_like(xt_pca_slices) * IMG_AUG_STD
        w_l = torch.where(yt_slices == 0, torch.tensor(NEG_WEIGHT).to(device), torch.tensor(1.0).to(device))
        l_probs = torch.sigmoid(m_ct(xt_clin_slices, True) + m_res(xt_pca_slices + i_aug, True))
        criterion(l_probs, yt_slices, weight=w_l).backward(); opt_res.step()

    # --- 3. Calibration & Test (Spectral) ---
    m_sp_wrap = ResidualWrapper(m_ct, m_res)
    with torch.no_grad():
        v_probs_ct, v_probs_sp, v_labels = [], [], []
        for s_id in unique_subjs[v_subj_idx]:
            s_mask = (tr_subj_map == s_id)
            v_c = torch.tensor(scaler_c.transform(X_tr_clin[s_mask]), dtype=torch.float32).to(device)
            v_p = torch.tensor(X_tr_pca[s_mask], dtype=torch.float32).to(device)
            v_probs_ct.append(m_ct(v_c).mean().cpu().item())
            v_probs_sp.append(m_sp_wrap(torch.cat([v_p, v_c], dim=1)).mean().cpu().item())
            v_labels.append(y_tr_slices[s_mask][0])

    fold_ths_ct.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_ct), np.array(v_labels), 'cpu'))
    fold_ths_sp.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_sp), np.array(v_labels), 'cpu'))

    with torch.no_grad():
        te_p_ct_f, te_p_sp_f = [], []
        for s_id in np.unique(te_subj_map):
            s_mask = (te_subj_map == s_id)
            t_c = torch.tensor(scaler_c.transform(X_te_clin[s_mask]), dtype=torch.float32).to(device)
            t_p = torch.tensor(X_te_pca[s_mask], dtype=torch.float32).to(device)
            te_p_ct_f.append(m_ct(t_c).mean().cpu().item())
            te_p_sp_f.append(torch.sigmoid(m_ct(t_c, True) + m_res(t_p, True)).mean().cpu().item())
        fold_preds_ct.append(te_p_ct_f); fold_preds_sp.append(te_p_sp_f)

    # --- 4. Spatial ViT (Baseline - Run LAST to avoid RNG corruption) ---
    # We save the RNG state before ViT and restore it after to ensure absolute isolation
    rng_state = torch.get_rng_state()
    
    m_spatial = SpatialViT().to(device)
    opt_spatial = optim.Adam(m_spatial.parameters(), lr=1e-4)
    xt_img_slices = torch.tensor(X_tr_slices[t_idx_slices]).view(-1, 1, 128, 128).float().to(device)

    

    for epoch in range(100):
        m_spatial.train(); opt_spatial.zero_grad()
        p_spatial = m_spatial(xt_img_slices + torch.randn_like(xt_img_slices)*IMG_AUG_STD)
        w_l = torch.where(yt_slices == 0, torch.tensor(NEG_WEIGHT).to(device), torch.tensor(1.0).to(device))
        criterion(p_spatial, yt_slices, weight=w_l).backward(); opt_spatial.step()

    m_spatial.eval()
    with torch.no_grad():
        v_p_spatial = []
        for s_id in unique_subjs[v_subj_idx]:
            s_mask = (tr_subj_map == s_id)
            v_img = torch.tensor(X_tr_slices[s_mask]).view(-1, 1, 128, 128).float().to(device)
            v_p_spatial.append(m_spatial(v_img).mean().cpu().item())
        fold_ths_spatial.append(calibrate_balanced(PassThrough(), None, np.array(v_p_spatial), np.array(v_labels), 'cpu'))

        te_p_spatial_f = []
        for s_id in np.unique(te_subj_map):
            s_mask = (te_subj_map == s_id)
            t_img = torch.tensor(X_te_slices[s_mask]).view(-1, 1, 128, 128).float().to(device)
            te_p_spatial_f.append(m_spatial(t_img).mean().cpu().item())
        fold_preds_spatial.append(te_p_spatial_f)
    
    torch.set_rng_state(rng_state)

# ============================================================
# RESULTS
# ============================================================
y_te_u = np.array([y_te_slices[te_subj_map == s][0] for s in np.unique(te_subj_map)])

# Simple LR Baselines

lr_c = LogisticRegression(class_weight='balanced', max_iter=5000, solver='lbfgs', tol=1e-4).fit(X_tr_clin, y_tr_slices)
p_lr_c = np.array([lr_c.predict_proba(X_te_clin[te_subj_map == s])[:, 1].mean() for s in np.unique(te_subj_map)])
th_lr_c = calibrate_balanced(PassThrough(), None, p_lr_c, y_te_u, 'cpu')

lr_j = LogisticRegression(class_weight='balanced', max_iter=5000, solver='lbfgs', tol=1e-4).fit(np.hstack([X_tr_pca, X_tr_clin]), y_tr_slices)
p_lr_j = np.array([lr_j.predict_proba(np.hstack([X_te_pca, X_te_clin])[te_subj_map == s])[:, 1].mean() for s in np.unique(te_subj_map)])
th_lr_j = calibrate_balanced(PassThrough(), None, p_lr_j, y_te_u, 'cpu')

# Aggregated Model Results
psp, p_spatial = np.mean(fold_preds_sp, axis=0), np.mean(fold_preds_spatial, axis=0)
th_sp, th_spatial = np.mean(fold_ths_sp), np.mean(fold_ths_spatial)

res_lr_c = get_m(y_te_u, (p_lr_c >= th_lr_c), p_lr_c)
res_lr_j = get_m(y_te_u, (p_lr_j >= th_lr_j), p_lr_j)
res_spatial = get_m(y_te_u, (p_spatial >= th_spatial), p_spatial)
res_sp = get_m(y_te_u, (psp >= th_sp), psp)

print(f"\n{'Metric':<12} | {'LR Clin':<10} | {'LR Joint':<10} | {'Spatial ViT':<12} | {'Spectral ViT':<15}")
print("-" * 80)
for i, name in enumerate(['Acc', 'Sens', 'Spec', 'AUC']):
    print(f"{name:<12} | {res_lr_c[i]:<10.3f} | {res_lr_j[i]:<10.3f} | {res_spatial[i]:<12.3f} | {res_sp[i]:<15.3f}")

from scipy.stats import wilcoxon
_, p_val = wilcoxon(np.abs(y_te_u - p_lr_j), np.abs(y_te_u - psp), alternative='greater')
print(f"\nWilcoxon p-value (Spectral vs Joint LR): {p_val:.4f}")

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)



Preparing MSW dataset (Forced 6-dim alignment) 
Pre-flight check: Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60

Final MSW Breakdown:
 - Unique Subjects on Disk: 111
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 45
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (45 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 21, 22, 24, 25, 27, 28, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Pre-flight check: Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Age     : 63.13
  > Sex     : 0.46
  > Dur     : 8.54